# Ruby Envelope Vault v1.0.0 — Full Kaggle / Colab Demo

**English / Tiếng Việt** · Rails 8 · Ruby 3.4 · PostgreSQL 17 · Redis · AES-256-GCM · Minitest / Selenium · synthetic demo

This notebook **clones the source from GitHub**, checks the release's exact commit, installs all required software through the repository's existing `bootstrap.sh`, runs full Rails/browser tests in a separate explicitly controlled-umask stage, seeds six categories of encrypted sample data, starts Rails, verifies HTTP routes, and can capture ten actual UI screenshots. A temporary, email-gated Cloudflare Quick Tunnel is optional for interactive browser access.

**Security boundaries:** Never use real credentials, live production keys, or production vault records. Notebook sessions, Kaggle outputs and tunnel URLs are temporary. GitHub repository is **Private**; enable Internet and supply `GH_READ_TOKEN` with read-only repository access (Kaggle Secrets / Colab Secrets). Do **not** publish notebook outputs containing login credentials, tunnel URLs, or other private metadata. This is a *source/demo* environment, not a production acceptance.

**Reference:** https://github.com/dangkhoa2016/Rails-8-Envelope-Encryption-Vault · The demo is checked out from the v1.0.0 annotated tag with source-file integrity fingerprints.

## 0. Settings / Cấu hình

Use **Kaggle CPU (Internet ON)**; Google Colab CPU is also supported on Ubuntu/Debian. An accelerated GPU adds no benefit. Avoid concurrent installations, and keep the notebook session active for the browser demo.

Set `ACCESS_EMAIL` to your real email address **only if you want a temporary browser URL** and an email one-time code gate. Leave it blank for local-only execution. Cloudflare quick tunnels are for testing, not production. A failed email gate must **not** silently expose an unprotected tunnel.

In [ ]:
from pathlib import Path
import os, sys, re, time, json, secrets, subprocess, tempfile, urllib.request

REPO_SLUG = "dangkhoa2016/Rails-8-Envelope-Encryption-Vault"
REF = "v1.0.0"
EXPECTED_SOURCE_BLOBS = {'bootstrap.sh': '0050129644d29964e94a9de29e7bf1a2f83db43c', 'Gemfile.lock': '64f35dc858be11d31ea2bed3f8fe74e10aa42c0b', 'script/bootstrap-services-lib.sh': 'a87e6c1077e773c9a951074d3beb483a2170964b'}
BASE = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else "/content")
REPO = BASE / "ruby-envelope-vault-demo-v1"
RUNTIME = BASE / ".ruby-envelope-vault-runtime"
ARTIFACTS = BASE / "ruby-envelope-vault-demo-evidence"
ARTIFACTS.mkdir(parents=True, exist_ok=True, mode=0o700)
ARTIFACTS.chmod(0o700)
PORT = 3000
ACCESS_EMAIL = ""  # e.g. "you@example.com"; optional: enable OTP-protected temporary tunnel
CAPTURE_SCREENSHOTS = True
MAKE_SHOWCASE_PACKAGE = True
DEMO_EMAIL = "showcase@example.test"
print("Platform:", "Kaggle" if str(BASE)=="/kaggle/working" else "Colab")
print("Source:", REPO_SLUG, REF)
print("Evidence directory:", ARTIFACTS)

## 1. Private GitHub checkout (no token in URL/log/history)

Public repositories require no token. For private repositories, the notebook first tries Kaggle's `GH_READ_TOKEN`, then Colab's secret with the same name. If not present, it prompts privately. A short-lived `GIT_ASKPASS` script passes the token in process environment only while cloning. The checkout is verified against the annotated tag, source-file Git blob fingerprints and VERSION before any execution; it will **not** overwrite an existing unrelated checkout.

In [ ]:
def read_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GH_READ_TOKEN")
        if value:
            return value
    except Exception:
        pass
    try:
        from google.colab import userdata
        value = userdata.get("GH_READ_TOKEN")
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get("GH_READ_TOKEN", "")
    if value:
        return value
    from getpass import getpass
    return getpass("GitHub private repo READ-ONLY token (input hidden): ")

def git(*args, cwd=None, env=None):
    return subprocess.run(["git", *args], cwd=cwd, env=env, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True).stdout.strip()

if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(f"Refusing to overwrite an existing non-Git directory: {REPO}")
    clone = ["git", "-c", "credential.helper=", "clone", "--depth", "1",
             "--branch", REF, "--single-branch",
             f"https://github.com/{REPO_SLUG}.git", str(REPO)]
    anonymous = subprocess.run(clone, stdout=subprocess.DEVNULL,
                               stderr=subprocess.PIPE, text=True,
                               env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if anonymous.returncode:
        # Public repositories work anonymously; private ones need read access.
        if REPO.exists():
            raise RuntimeError("Clone left an incomplete directory; remove it before retrying")
        token = read_github_token()
        if not token:
            raise RuntimeError("Private repository requires GH_READ_TOKEN with Contents: Read")
        with tempfile.TemporaryDirectory(prefix="rv-git-auth-", dir=str(BASE)) as authdir:
            askpass = Path(authdir) / "askpass.sh"
            askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf "%s" "x-access-token";; *Password*) printf "%s" "$RV_GITHUB_TOKEN";; *) exit 1;; esac\n')
            askpass.chmod(0o700)
            env = os.environ.copy()
            env.update({"RV_GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass),
                        "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_NOSYSTEM": "1"})
            try:
                subprocess.run(clone, env=env, stdout=subprocess.DEVNULL,
                               stderr=subprocess.PIPE, text=True, check=True)
            finally:
                token = None
sha = git("rev-parse", "HEAD", cwd=REPO)
tag_commit = git("rev-parse", f"{REF}^{{commit}}", cwd=REPO)
if sha != tag_commit:
    raise RuntimeError(f"Release checkout does not match annotated tag {REF}: {sha} != {tag_commit}")
for path, expected_blob in EXPECTED_SOURCE_BLOBS.items():
    actual_blob = git("rev-parse", f"HEAD:{path}", cwd=REPO)
    if actual_blob != expected_blob:
        raise RuntimeError(f"Release source integrity mismatch for {path}: expected {expected_blob}, got {actual_blob}")
if (REPO / "VERSION").read_text().strip() != "1.0.0":
    raise RuntimeError("Release VERSION mismatch")
if git("status", "--porcelain", cwd=REPO):
    raise RuntimeError("Source checkout is not clean; use a fresh notebook directory")
print("SOURCE_VERIFIED=PASS", sha)

## 2. Complete bootstrap / Cài đặt và kiểm thử đầy đủ

This delegates to the **existing repository bootstrap**; it does not reimplement database/KEK provisioning. It installs packages, Ruby 3.4 via mise, PostgreSQL, Redis, required gems and assets, prepares app/test databases and a local demo KEK, then runs crypto/security self-tests and `project-doctor`. **Next**, the notebook separately executes the **full Rails + Selenium suites** with explicit test-process `umask 022` (matching GitHub CI), because v1.0.0's `PrepareSecretEnvTest` uses `Dir.mkdir(..., 0755)` and assumes a permissive inherited umask; the bootstrap intentionally sets `umask 077` to protect secret files. Only the isolated test process uses `022`; secret files remain mode `0600`. This documents a test portability issue without changing the frozen v1.0.0 source.

The first fresh run may involve downloading/compiling dependencies. **Do not execute this cell twice while it is still running.** For failures, inspect the log privately and troubleshoot the specific failing stage; never paste its entire contents without redaction.

In [ ]:
BOOTSTRAP_LOG = ARTIFACTS / "bootstrap.log"
base_env = os.environ.copy()
base_env.update({"START_SERVER":"0", "RUN_TESTS":"0", "CREATE_OWNER":"0",
                 "ALLOW_SIGNUPS":"false", "DEMO_MODE":"true", "REQUIRE_MFA":"false",
                 "REAUTH_ON_REVEAL":"true", "PORT":str(PORT), "INSTALL_AGE":"1"})
with BOOTSTRAP_LOG.open("w", encoding="utf-8") as log:
    os.chmod(BOOTSTRAP_LOG, 0o600)
    cp = subprocess.run(["bash", "./bootstrap.sh", "setup"], cwd=REPO,
                        env=base_env, stdout=log, stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL)
if cp.returncode:
    raise RuntimeError(f"Bootstrap failed with exit={cp.returncode}. Examine PRIVATE log: {BOOTSTRAP_LOG}")
print("BOOTSTRAP=PASS", "crypto_security_doctor=completed", "log:", BOOTSTRAP_LOG)

# Full Rails + Selenium suites remain mandatory: controlled test-only umask 022.
# v1.0.0 test/prepare_secret_env_test.rb assumes a 0755 Dir.mkdir output.
TESTS_LOG = ARTIFACTS / "full-tests.log"
test_script = """source ./activate-runtime.sh
umask 022
RAILS_ENV=test bin/rails zeitwerk:check &&
RAILS_ENV=test bin/rails db:prepare &&
RAILS_ENV=test bin/rails test &&
RAILS_ENV=test bin/rails test:system
"""
with TESTS_LOG.open("w",encoding="utf-8") as log:
    os.chmod(TESTS_LOG,0o600)
    tests = subprocess.run(["bash","-e","-c",test_script],cwd=REPO,env=base_env,
                           stdout=log,stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL)
if tests.returncode:
    raise RuntimeError(f"Full Rails/Selenium test run failed (exit={tests.returncode}); private log: {TESTS_LOG}")
print("FULL_RAILS_TESTS=PASS", "RAILS_SYSTEM_TESTS=PASS", "test_umask=022", "log:", TESTS_LOG)

## 3. Encrypted synthetic sample / Tạo dữ liệu mã hóa

Create or reuse a **temporary demo-only password** (14+ characters), saved outside Git with mode `0600`. The notebook displays the login password only for the current private interactive session. **Clear cell output before publishing this notebook.** The app's existing `demo:fixture` task seeds Login, API Key, Environment, Database, Secure Note and SSH items, including version history. Signups are disabled.

For a fully unattended published run, clear the private password output before publishing. Never commit a real user password.

In [ ]:
password_file = RUNTIME / "tmp" / "notebook-demo-password"
password_file.parent.mkdir(parents=True, exist_ok=True, mode=0o700)
password_file.parent.chmod(0o700)
if password_file.exists():
    demo_password = password_file.read_text().strip()
else:
    demo_password = secrets.token_urlsafe(22)
    fd = os.open(password_file, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    with os.fdopen(fd, "w") as f:
        f.write(demo_password)
password_file.chmod(0o600)
if len(demo_password) < 14:
    raise RuntimeError("Demo password too short")
fixture_env = base_env.copy()
fixture_env.update({"DEMO_MODE":"true", "ALLOW_SIGNUPS":"false", "DEMO_PASSWORD_FILE":str(password_file),
                    "DEMO_EMAIL":DEMO_EMAIL, "DEMO_RESET":"false"})
fixture_cmd = 'source ./activate-runtime.sh && exec bin/rails demo:fixture'
cp = subprocess.run(["bash", "-c", fixture_cmd], cwd=REPO, env=fixture_env,
                    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if cp.returncode or "DEMO_FIXTURE=PASS" not in cp.stdout:
    raise RuntimeError("Fixture failed; inspect runtime configuration and Rails environment privately")
print(cp.stdout.strip())  # safe fixture summary only: no vault secret contents
print("LOGIN_EMAIL:", DEMO_EMAIL)
print("TEMPORARY_DEMO_PASSWORD (PRIVATE NOTEBOOK OUTPUT; CLEAR BEFORE SHARING):", demo_password)

## 4. Optional email-gated Cloudflare tunnel / Mở giao diện ngoài notebook

**Only runs if `ACCESS_EMAIL` is set.** Tunnel URLs are ephemeral; Cloudflare Quick Tunnels are development-only. `--allowed-mail` gates access via an email one-time code. If the command fails, this cell stops instead of falling back to an openly accessible tunnel. You must not publish real data into this demo. Rails will allow **only the specific generated external host**, via `RAILS_DEVELOPMENT_HOSTS` (Rails 8 standard configuration).

In [ ]:
tunnel_proc = None
tunnel_url = None
tunnel_hostname = None
if ACCESS_EMAIL.strip():
    if not re.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", ACCESS_EMAIL.strip()):
        raise ValueError("Provide a real email for the OTP access gate")
    import platform
    cloudflared = RUNTIME / "bin" / "cloudflared"
    cloudflared.parent.mkdir(parents=True, exist_ok=True)
    if not cloudflared.is_file():
        if platform.machine().lower() not in ("x86_64", "amd64"):
            raise RuntimeError("Automatic cloudflared binary download currently supports linux-amd64 only")
        src="https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        tmp=cloudflared.with_suffix(".partial")
        urllib.request.urlretrieve(src, tmp)
        if tmp.open("rb").read(4) != b"\x7fELF":
            raise RuntimeError("Downloaded cloudflared executable has invalid ELF signature")
        tmp.chmod(0o700)
        tmp.replace(cloudflared)
    cloudflared.chmod(0o700)
    tunnel_log = ARTIFACTS / "cloudflared.log"
    f = tunnel_log.open("w", encoding="utf-8")
    os.chmod(tunnel_log, 0o600)
    tunnel_proc = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate",
                                    "--url", f"http://127.0.0.1:{PORT}",
                                    "--allowed-mail", ACCESS_EMAIL.strip()],
                                   stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
    f.close()
    pat=re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    for _ in range(60):
        data=tunnel_log.read_text(errors="replace")
        m=pat.search(data)
        if m:
            tunnel_url=m.group(0)
            tunnel_hostname=tunnel_url.split("//",1)[1]
            break
        if tunnel_proc.poll() is not None:
            raise RuntimeError("Email-gated tunnel did not start; inspect private cloudflared log. No open fallback.")
        time.sleep(2)
    if not tunnel_url:
        tunnel_proc.terminate()
        raise RuntimeError("Could not obtain the email-gated tunnel URL")
    print("EMAIL_GATED_TUNNEL=READY", tunnel_url, "allowed_mail:", ACCESS_EMAIL.strip())
else:
    print("TUNNEL=OFF (local-only). Set ACCESS_EMAIL above to enable an email-gated browser URL.")

## 5. Start Rails and validate the actual HTTP app / Khởi chạy ứng dụng

Bootstrap already started PostgreSQL/Redis. The app is started from `bootstrap.sh start` to preserve the supported runtime lifecycle. The Rails process and logs persist while the notebook session is alive. The health endpoint is checked over **real HTTP**, as is the login form.

In [ ]:
rails_log = ARTIFACTS / "rails.log"
rails_proc = globals().get("rails_proc")
if rails_proc is None or rails_proc.poll() is not None:
    rails_env = fixture_env.copy()
    rails_env.update({"START_SERVER":"1", "PORT":str(PORT)})
    if tunnel_hostname:
        rails_env["RAILS_DEVELOPMENT_HOSTS"] = tunnel_hostname
    f=rails_log.open("w", encoding="utf-8")
    os.chmod(rails_log, 0o600)
    rails_proc=subprocess.Popen(["bash", "./bootstrap.sh", "start"], cwd=REPO,
                                env=rails_env, stdout=f, stderr=subprocess.STDOUT,
                                stdin=subprocess.DEVNULL, start_new_session=True)
    f.close()
local_url=f"http://127.0.0.1:{PORT}"
health=None
for _ in range(100):
    try:
        with urllib.request.urlopen(local_url+"/up",timeout=3) as x:
            health=x.status
        if health==200: break
    except Exception:
        pass
    if rails_proc.poll() is not None:
        raise RuntimeError(f"Rails exited unexpectedly. Check protected log: {rails_log}")
    time.sleep(2)
if health!=200:
    raise RuntimeError(f"Local Rails health failed; inspect protected log: {rails_log}")
with urllib.request.urlopen(local_url+"/session/new",timeout=10) as resp:
    login_status=resp.status
    login_html=resp.read(200_000).decode("utf-8","replace")
assert login_status==200 and ("password" in login_html.lower())
print("RAILS_HEALTH=PASS /up=200")
print("LOGIN_FORM=PASS /session/new=200")
print("LOCAL_URL:",local_url+"/session/new")
if tunnel_url:
    print("BROWSER_DEMO_URL (OTP email gate required):",tunnel_url+"/session/new")

## 6. Database and envelope acceptance / Kiểm tra dữ liệu đã mã hóa

Read only non-secret counts and envelope metadata. No plaintext secret is printed. Check that the six examples have encrypted payloads, 12-byte GCM nonces, 16-byte authentication tags and wrapped DEKs.

In [ ]:
ruby_src = r"""
require "json"
user = User.find_by!(email_address: ENV.fetch("DEMO_EMAIL"))
items = user.vault_items.includes(:versions).to_a
versions = items.flat_map(&:versions)
raise "expected 6 sample items" unless items.size == 6
raise "missing immutable versions" unless versions.size >= 7
raise "invalid ciphertext envelope" unless versions.all? { |v|
  v.cipher_suite == "AES-256-GCM" && v.nonce.bytesize == 12 &&
  v.authentication_tag.bytesize == 16 && v.ciphertext.bytesize > 0 &&
  v.wrapped_dek.bytesize > 0
}
puts "ENCRYPTED_FIXTURES=PASS #{JSON.generate(items: items.size, versions: versions.size,
  kinds: items.map(&:kind).sort, cipher: 'AES-256-GCM', aead_nonce_bytes: 12, auth_tag_bytes: 16)}"
"""
with tempfile.NamedTemporaryFile(mode="w",suffix=".rb",dir=str(ARTIFACTS),delete=False) as f:
    f.write(ruby_src)
    rubyfile=Path(f.name)
rubyfile.chmod(0o600)
try:
    env = fixture_env.copy()
    proc=subprocess.run(["bash","-c",f'source ./activate-runtime.sh && exec bin/rails runner "{rubyfile}"'],
                        cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    if proc.returncode: raise RuntimeError("Encrypted fixture verification failed; inspect Rails logs privately")
    print(proc.stdout.strip())
finally:
    rubyfile.unlink(missing_ok=True)

## 7. Capture ten real Selenium screenshots / Ảnh giao diện thật

**Optional but enabled by default.** Runs the repository's existing `script/capture-showcase` with a separate test database, preserving live development-demo records. It captures desktop and mobile pages, hashes each PNG, and writes a metadata manifest. It is **not** a synthetic mock of the interface. The test may need a rerun if Chrome/Selenium encounters an intermittent browser timing failure.

In [ ]:
SCREENSHOTS = ARTIFACTS / "screenshots"
if CAPTURE_SCREENSHOTS:
    capture_log=ARTIFACTS / "showcase-capture.log"
    screen_env=fixture_env.copy()
    screen_env["SHOWCASE_OUTPUT"]=str(SCREENSHOTS)
    cmd='source ./activate-runtime.sh && exec bash ./script/capture-showcase'
    with capture_log.open("w",encoding="utf-8") as f:
        os.chmod(capture_log,0o600)
        cp=subprocess.run(["bash","-c",cmd],cwd=REPO,env=screen_env,
                          stdout=f,stderr=subprocess.STDOUT)
    if cp.returncode: raise RuntimeError(f"Screenshot capture failed; private log: {capture_log}")
    shots=sorted(SCREENSHOTS.glob("*.png"))
    if len(shots)!=10: raise RuntimeError(f"Expected 10 screenshots, received {len(shots)}")
    if not (SCREENSHOTS/"SHA256SUMS.txt").is_file() or not (SCREENSHOTS/"MANIFEST.tsv").is_file():
        raise RuntimeError("Screenshot integrity metadata missing")
    print(f"SHOWCASE_CAPTURE=PASS screenshots={len(shots)}")
else:
    print("SHOWCASE_CAPTURE=SKIPPED (user setting)")

## 8. Package the sanitized showcase / Đóng gói

The repository's self-contained packaging and SHA-256 validators are executed against actual screenshots. Unlike a source CI badge, this proves the PNG/manifests/docs produced by **this notebook run** have been validated. Do not add env/KEKs, database dumps or notebook logs to the public bundle.

In [ ]:
SHOWCASE_OUT = ARTIFACTS / "showcase-package"
if MAKE_SHOWCASE_PACKAGE and CAPTURE_SCREENSHOTS:
    pkg_env=fixture_env.copy()
    pkg_env.update({"SHOWCASE_DIR":str(SCREENSHOTS),"PUBLIC_SHOWCASE_OUTPUT_DIR":str(SHOWCASE_OUT)})
    pkg_log=ARTIFACTS / "showcase-package.log"
    with pkg_log.open("w",encoding="utf-8") as f:
        os.chmod(pkg_log,0o600)
        cp=subprocess.run(["bash","./script/build-public-showcase"],cwd=REPO,
                          env=pkg_env,stdout=f,stderr=subprocess.STDOUT)
    if cp.returncode: raise RuntimeError(f"Package build failed; private log: {pkg_log}")
    print("PUBLIC_SHOWCASE_PACKAGE=PASS")
    print("Sanitized package outputs:", [p.name for p in SHOWCASE_OUT.iterdir() if p.is_file()])
else:
    print("PUBLIC_SHOWCASE_PACKAGE=SKIPPED")

## 9. Visual gallery and evidence / Tổng kết

This final optional preview displays only the generated synthetic-data screenshots, never real secrets. The actual browser site remains live until the session ends or you run the shutdown cell.

In [ ]:
if CAPTURE_SCREENSHOTS:
    from IPython.display import display
    from PIL import Image,ImageOps,ImageDraw
    thumbs=[]
    for path in sorted(SCREENSHOTS.glob("*.png")):
        im=Image.open(path).convert("RGB")
        im.thumbnail((420,300))
        tile=Image.new("RGB",(450,350),"white")
        tile.paste(im,((450-im.width)//2,8))
        ImageDraw.Draw(tile).text((12,322),path.stem[:52],fill="black")
        thumbs.append(tile)
    if thumbs:
        grid=Image.new("RGB",(450*2,350*5),"#dddddd")
        for idx,t in enumerate(thumbs): grid.paste(t,((idx%2)*450,(idx//2)*350))
        display(grid)
print("SOURCE_SHA:",sha)
print("DEMO_EMAIL:",DEMO_EMAIL)
print("LOCAL_URL:",local_url)
if tunnel_url: print("OTP_GATED_URL:",tunnel_url)
print("PRIVATE_LOG_DIRECTORY:",ARTIFACTS)
print("Do not publish notebook cell outputs that contain demo passwords or private run logs.")

## 10. Optional shutdown / Dừng demo

Run only when finished viewing the app. Terminates notebook-owned Rails and Cloudflare processes; PostgreSQL/Redis remain managed by the repository until stopped. This cell is **not** auto-executed so interactive preview remains available.

In [ ]:
# Run manually when you are finished (uncomment):
# if tunnel_proc is not None and tunnel_proc.poll() is None: tunnel_proc.terminate()
# if rails_proc is not None and rails_proc.poll() is None: rails_proc.terminate()
# subprocess.run(["bash", "./bootstrap.sh", "stop"], cwd=REPO, check=False)
# print("DEMO_STOPPED")